# Module C0 — Architecture multi-fichiers

**Phase 3 — Savoirs transversaux**  
Compatible Google Colab : toutes les données sont définies inline, aucun fichier externe requis.

## Section 1 — Rappels synthétiques

### Principe de base
Un programme bien conçu découpe son code en **modules** : chaque fichier a **une seule responsabilité**.

### Les 6 fichiers de Drone Rescue

| Fichier | Responsabilité | Importe |
|---|---|---|
| `config.py` | Constantes du jeu | *(rien)* |
| `logique.py` | Règles métier | `config` |
| `affichage.py` | Rendu terminal | `config` |
| `logger.py` | Écriture des logs | *(rien)* |
| `console.py` | Boucle de jeu, saisie | `logique`, `affichage`, `logger` |
| `main.py` | Point d'entrée | `console`, `logique`, `logger` |

### Règle d'or
Les dépendances vont **du général au spécifique**. `main` → `console` → `logique/affichage`. **Jamais l'inverse.**

### Rappel du module précédent (P09)
Dans P09, tu as assemblé tous les modules dans `main.py`. Tu as écrit `if __name__ == '__main__': main()` pour isoler l'exécution.

## Section 2 — Exercice guidé : graphe de dépendances

Dans cet exercice, tu vas :
1. Représenter le graphe de dépendances sous forme de dictionnaire Python
2. Identifier les violations potentielles
3. Vérifier qu'une liste d'imports est licite

### Étape 1 — Représenter le graphe comme un dictionnaire

In [ ]:
# Graphe de dépendances : clé = module, valeur = liste des modules qu'il importe
# TODO : compléter les valeurs manquantes en te basant sur le tableau de rappel
graphe = {
    'config':    [],               # config n'importe rien
    'logique':   ['config'],       # logique importe config
    'affichage': ['config'],       # affichage importe config
    'logger':    [],               # TODO : que doit importer logger ?
    'console':   [],               # TODO : quels modules console importe-t-il ?
    'main':      [],               # TODO : quels modules main importe-t-il ?
}

### Étape 2 — Vérifier : le graphe est-il acyclique ?

In [ ]:
def detecter_cycle(graphe, depart, visite=None, pile=None):
    """Détecte un cycle dans un graphe de dépendances (DFS)."""
    if visite is None:
        visite = set()
    if pile is None:
        pile = set()
    visite.add(depart)
    pile.add(depart)
    for voisin in graphe.get(depart, []):
        if voisin not in visite:
            if detecter_cycle(graphe, voisin, visite, pile):
                return True
        elif voisin in pile:
            return True
    pile.discard(depart)
    return False

# Vérification
for module in graphe:
    if detecter_cycle(graphe, module):
        print(f'⚠️  Cycle détecté à partir de : {module}')
    else:
        print(f'✅ Aucun cycle depuis : {module}')

### Étape 3 — Identifier les violations : ces imports sont-ils valides ?

In [ ]:
# Liste d'imports à analyser
# Pour chaque import, réponds : VALIDE ou VIOLATION (et pourquoi)
imports_a_analyser = [
    ('console', 'logique'),      # console importe logique
    ('logique', 'console'),      # logique importe console
    ('affichage', 'logique'),    # affichage importe logique
    ('main', 'affichage'),       # main importe affichage
    ('config', 'main'),          # config importe main
]

def est_valide(graphe_ref, importeur, importe):
    """Vérifie qu'un import ne crée pas de dépendance remontante."""
    # TODO : retourner True si importe figure dans graphe_ref[importeur]
    #        ou si importe est une dépendance transitive valide
    # Indice : utilise graphe_ref[importeur] pour vérifier directement
    pass

# Test manuel
for importeur, importe in imports_a_analyser:
    valide = importe in graphe.get(importeur, []) or any(
        importe in graphe.get(dep, []) for dep in graphe.get(importeur, [])
    )
    statut = '✅ VALIDE' if valide else '❌ VIOLATION'
    print(f'{statut} : {importeur} → {importe}')

### Étape 4 — Tester `if __name__ == '__main__':`

In [ ]:
# Dans un notebook, __name__ vaut '__main__' — le bloc s'exécute toujours
# Dans un fichier importé, __name__ vaut le nom du fichier — le bloc est ignoré

print(f'Valeur de __name__ dans ce notebook : {__name__}')

# Simulation : que se passe-t-il si on importe un module qui a du code au niveau global ?
# Bonne pratique : tout code d'exécution doit être dans if __name__ == '__main__'

# TODO : écrire une fonction main_simulee() qui affiche 'Partie lancée !'
# puis l'appeler uniquement si __name__ == '__main__'

# TODO
def main_simulee():
    pass

if __name__ == '__main__':
    main_simulee()

In [ ]:
# Vérification finale
assert 'logique' in graphe['console'], "console doit importer logique"
assert 'affichage' in graphe['console'], "console doit importer affichage"
assert 'console' not in graphe['logique'], "logique NE DOIT PAS importer console"
assert graphe['config'] == [], "config ne doit rien importer"
print('✅ Graphe de dépendances correct !')

➡️ **Corrigé** : voir `C0_corrige.py`  
Module suivant : **C1 — Configuration externe**